In [3]:
import json
import os
import shutil
from pathlib import Path

In [ ]:
BOP_TEST_DIR = Path("../data/ycbv/test")
OUTPUT_DIR = Path("../data/yolo_dataset/test")

In [13]:
TARGET_IDS = {
    1: 0,   # master_chef_can
    2: 1,   # cracker_box
    4: 2,   # tomato_soup_can
    5: 3,   # mustard_bottle
    10: 4,  # banana
    12: 5,  # bleach_cleanser
    15: 6,  # power_drill
    17: 7   # scissors
}

IMG_WIDTH = 640
IMG_HEIGHT = 480

In [14]:
def setup_directories():
    os.makedirs(OUTPUT_DIR / "images", exist_ok=True)
    os.makedirs(OUTPUT_DIR / "labels", exist_ok=True)

In [15]:
def convert_bop_to_yolo():
    setup_directories()
    
    # BOP splits data into "scenes" (e.g., 000048, 000049)
    scene_dirs = [d for d in BOP_TEST_DIR.iterdir() if d.is_dir()]
    
    for scene in scene_dirs:
        scene_id = scene.name
        gt_file = scene / "scene_gt.json"
        info_file = scene / "scene_gt_info.json"
        
        if not gt_file.exists() or not info_file.exists():
            continue
            
        with open(gt_file, 'r') as f:
            scene_gt = json.load(f)
        with open(info_file, 'r') as f:
            scene_info = json.load(f)
            
        for img_id_str, annotations in scene_gt.items():
            img_info = scene_info[img_id_str]
            valid_labels = []
            
            for idx, ann in enumerate(annotations):
                obj_id = ann["obj_id"]
                
                # If it's one of our 8 target classes
                if obj_id in TARGET_IDS:
                    yolo_class_id = TARGET_IDS[obj_id]
                    
                    # BOP bbox format is [x_min, y_min, width, height]
                    bbox = img_info[idx]["bbox_visib"] 
                    x_min, y_min, w, h = bbox
                    
                    # YOLO requires normalized [x_center, y_center, width, height]
                    x_center = (x_min + w / 2) / IMG_WIDTH
                    y_center = (y_min + h / 2) / IMG_HEIGHT
                    norm_w = w / IMG_WIDTH
                    norm_h = h / IMG_HEIGHT
                    
                    valid_labels.append(f"{yolo_class_id} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}")
            
            # Only save if the image contains at least one of our target objects
            if valid_labels:
                img_name = f"{int(img_id_str):06d}.png"  # BOP images are padded to 6 zeros
                src_img_path = scene / "rgb" / img_name
                
                if src_img_path.exists():
                    # Create a unique filename combining scene and image ID
                    unique_name = f"scene{scene_id}_{img_name}"
                    
                    # Copy Image
                    shutil.copy(
                        src_img_path, 
                        OUTPUT_DIR / "images" / unique_name
                    )
                    
                    # Write YOLO Label
                    label_path = OUTPUT_DIR / "labels" / unique_name.replace(".png", ".txt")
                    with open(label_path, 'w') as label_file:
                        label_file.write("\n".join(valid_labels))

In [16]:
print("Filtering BOP dataset for 8 target classes...")
convert_bop_to_yolo()
print(f"Done! Clean YOLO dataset is ready at: {OUTPUT_DIR}")

Filtering BOP dataset for 8 target classes...
Done! Clean YOLO dataset is ready at: ../data/yolo/test
